13/07/2026

This notebook was created to prepare data for manual inspection, specifically for updating or constructing a new training dataset based on feedback from previous models.

- reads vector layers directly from PostGIS
- postgis username and password are saved in environment variables

In [1]:
import os
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Connect to PostGIS


Example of query:

`SELECT *` means all attributes, instead you can select which attributes to read

`als_ml_rezultati` is the selected **Schema**
`adaf_irish` is the desired **Table**

```SQL
'SELECT * FROM als_ml_rezultati.adaf_irish'
```

Another example:

```SQL
SELECT
    label_id,
    label_origin,
    annotation_round,
    source_model,
    geometry
FROM training_data.barrow_labels
WHERE label_origin = 'model_assisted'
```


In [2]:

connection_url = URL.create(
    drivername="postgresql",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host="gisko.zrc-sazu.si",
    port=15432,
    database="STONE_delovno"
)

engine = create_engine(connection_url)

In [3]:
example_gdf = gpd.read_postgis(
    'SELECT * FROM als_ml_rezultati.adaf_irish',
    engine
)

example_gdf.head()

,fid,geom,label,roundness,results_id
0,30,"MULTIPOLYGON (((245152 4759663.5, 245152 47596...",barrow,0.668456,0
1,40,"MULTIPOLYGON (((244371.5 4767340, 244372.5 476...",barrow,0.668456,0
2,3680,"MULTIPOLYGON (((264148.5 4768702, 264147.5 476...",barrow,0.668456,0
3,60,"MULTIPOLYGON (((245505.5 4762162, 245506 47621...",barrow,0.668456,0
4,141,"MULTIPOLYGON (((242989.5 4766970, 242990 47669...",barrow,0.668456,0


# Load supplementary vectors

Load the reference layer and tiles footprints. Refererence layer tells us which vectors are already labeled, tiles footprints will limit it only to the areas that are requiered for training.

In [4]:
# Footprint of the tiles
tiles_footprint = gpd.read_postgis(
    'SELECT geom FROM als_ml_podatki.learning_samples_v5_512px',
    engine
).dissolve()

In [5]:
# Reference data
ref_data = gpd.read_postgis(
    'SELECT * FROM als_podatki."BiH_ALS_interpretation"',
    engine
)

This is how we filter the labels only to our area of interest:

In [6]:
ref_data_filtered = ref_data[
    ref_data.geometry.intersects(tiles_footprint.geometry.iloc[0])
].copy()

print(f"Original features: {len(ref_data)}")
print(f"Intersecting features: {len(ref_data_filtered)}")

Original features: 13031
Intersecting features: 8846


# Load all the results from existing models

In [7]:
def load_result_layers(
    engine,
    schema,
    table_names,
    geom_col="geom",
):
    """
    Read several PostGIS vector layers into a dictionary of GeoDataFrames.

    Returns
    -------
    dict
        Keys are table names and values are GeoDataFrames.
    """
    result_layers = {}

    for table_name in table_names:

        # Double quotes are required for identifiers containing hyphens
        query = f'''
            SELECT "{geom_col}"
            FROM "{schema}"."{table_name}"
        '''

        gdf = gpd.read_postgis(
            query,
            engine,
            geom_col=geom_col,
        )

        gdf = (
            gdf[
                gdf.geometry.intersects(tiles_footprint.geometry.iloc[0])
            ]
            .copy()
            .reset_index(drop=True)
        )

        result_layers[table_name] = gdf

        print(
            f"Loaded {table_name}: "
            f"{len(gdf):,} features, CRS={gdf.crs}"
        )

    return result_layers

Let's see what layers are available on the database:

In [8]:
from sqlalchemy import inspect

schema = "als_ml_rezultati"

inspector = inspect(engine)
tables = inspector.get_table_names(schema=schema)
tables

['adaf_retrained_2-512px_luka',
 'adaf_irish',
 'adaf_retrained_1',
 'adaf_retrained_2-256px',
 'adaf_bih_0-256px',
 'adaf_bih_0-128px',
 'adaf_bih_0-512px',
 'adaf_retrained_2-512px',
 'possible_missed']

Load all seven result layers:

In [9]:
# PostgreSQL schema containing model results
results_schema = "als_ml_rezultati"

# Exact PostgreSQL table names
result_table_names = [
    "adaf_irish",
    "adaf_retrained_1",
    "adaf_retrained_2-256px",
    "adaf_bih_0-256px",
    "adaf_bih_0-128px",
    "adaf_bih_0-512px",
    "adaf_retrained_2-512px",
]

result_layers = load_result_layers(
    engine=engine,
    schema=results_schema,
    table_names=result_table_names,
    geom_col="geom",
)

Loaded adaf_irish: 3,827 features, CRS=EPSG:32634
Loaded adaf_retrained_1: 8,311 features, CRS=EPSG:32634
Loaded adaf_retrained_2-256px: 7,629 features, CRS=EPSG:32634
Loaded adaf_bih_0-256px: 8,018 features, CRS=EPSG:32634
Loaded adaf_bih_0-128px: 13,842 features, CRS=EPSG:32634
Loaded adaf_bih_0-512px: 7,492 features, CRS=EPSG:32634
Loaded adaf_retrained_2-512px: 8,035 features, CRS=EPSG:32634


# Function for finding candidates

In [24]:
import re

import geopandas as gpd
import pandas as pd


def _clean_column_name(name):
    """Convert a result name into a database-safe column name."""
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", name)
    return f"src_{name.strip('_').lower()}"


def _union_all(geometry):
    """Compatibility helper for different GeoPandas versions."""
    try:
        return geometry.union_all()
    except AttributeError:
        return geometry.unary_union


def create_missed_label_candidates(
    ref_gdf,
    result_gdfs,
):
    """
    Identify possible labels missing from the reference dataset.

    Parameters
    ----------
    ref_gdf : geopandas.GeoDataFrame
        Existing reference labels.

    result_gdfs : dict
        Dictionary in the form:

        {
            "result_name": result_geodataframe,
            ...
        }

    Returns
    -------
    geopandas.GeoDataFrame
        Candidate missed labels. Overlapping detections from different
        result layers are merged into one object. Boolean source fields
        indicate which result layers detected each object.
    """

    if ref_gdf.crs is None:
        raise ValueError("ref_gdf does not have a defined CRS.")

    if not result_gdfs:
        raise ValueError("result_gdfs is empty.")

    target_crs = ref_gdf.crs

    # ---------------------------------------------------------
    # 1. Prepare reference labels
    # ---------------------------------------------------------

    reference = ref_gdf[[ref_gdf.geometry.name]].copy()

    reference = reference[
        reference.geometry.notna()
        & ~reference.geometry.is_empty
    ].copy()

    if not reference.empty:
        try:
            reference.geometry = reference.geometry.make_valid()
        except AttributeError:
            reference.geometry = reference.geometry.buffer(0)

        reference_union = _union_all(reference.geometry)
    else:
        reference_union = None

    # Store predictions remaining after reference filtering
    filtered_results = {}

    # Store all remaining polygons in one combined GeoDataFrame
    candidate_parts = []

    # Map original result names to output column names
    source_columns = {}

    # ---------------------------------------------------------
    # 2. Remove detections already represented in reference data
    # ---------------------------------------------------------

    for result_name, result_gdf in result_gdfs.items():

        if result_gdf.crs is None:
            raise ValueError(
                f"Result layer '{result_name}' does not have a defined CRS."
            )

        result = result_gdf[[result_gdf.geometry.name]].copy()

        if result.crs != target_crs:
            result = result.to_crs(target_crs)

        result = result[
            result.geometry.notna()
            & ~result.geometry.is_empty
        ].copy()

        try:
            result.geometry = result.geometry.make_valid()
        except AttributeError:
            result.geometry = result.geometry.buffer(0)

        # A prediction intersecting an existing reference polygon is
        # considered an already-known object and is therefore removed.
        if reference_union is not None:
            result = result[
                ~result.geometry.intersects(reference_union)
            ].copy()

        result = result.reset_index(drop=True)

        filtered_results[result_name] = result

        source_column = _clean_column_name(result_name)
        source_columns[result_name] = source_column

        if not result.empty:
            result["source_result"] = result_name
            candidate_parts.append(result)

    # ---------------------------------------------------------
    # 3. Handle the case where no possible missed labels remain
    # ---------------------------------------------------------

    if not candidate_parts:
        columns = {
            "candidate_id": pd.Series(dtype="int64"),
            "source_count": pd.Series(dtype="int64"),
            "source_results": pd.Series(dtype="object"),
            "geom": gpd.GeoSeries([], crs=target_crs),
        }

        for column_name in source_columns.values():
            columns[column_name] = pd.Series(dtype="bool")

        return gpd.GeoDataFrame(
            columns,
            geometry="geom",
            crs=target_crs,
        )

    all_candidates = gpd.GeoDataFrame(
        pd.concat(candidate_parts, ignore_index=True),
        geometry="geom",
        crs=target_crs,
    )

    # ---------------------------------------------------------
    # 4. Merge detections that intersect each other
    # ---------------------------------------------------------
    #
    # dissolve() unions all geometries.
    # explode() separates disconnected objects again.
    #
    # Therefore:
    # - overlapping polygons become one candidate;
    # - polygons connected through a chain of intersections also become
    #   one candidate;
    # - spatially separate polygons remain separate candidates.
    # ---------------------------------------------------------

    candidates = (
        all_candidates[[all_candidates.geometry.name]]
        .dissolve()
        .explode(index_parts=False)
        .reset_index(drop=True)
    )

    candidates["candidate_id"] = range(1, len(candidates) + 1)

    # ---------------------------------------------------------
    # 5. Add Boolean source fields
    # ---------------------------------------------------------

    for result_name, source_column in source_columns.items():

        candidates[source_column] = False

        source_gdf = filtered_results[result_name]

        if source_gdf.empty:
            continue

        intersections = gpd.sjoin(
            candidates[["candidate_id", candidates.geometry.name]],
            source_gdf[[source_gdf.geometry.name]],
            how="inner",
            predicate="intersects",
        )

        detected_candidate_ids = intersections["candidate_id"].unique()

        candidates.loc[
            candidates["candidate_id"].isin(detected_candidate_ids),
            source_column,
        ] = True

    # ---------------------------------------------------------
    # 6. Add combined source information
    # ---------------------------------------------------------

    def get_source_names(row):
        names = [
            result_name
            for result_name, source_column in source_columns.items()
            if row[source_column]
        ]
        return "; ".join(names)

    candidates["source_results"] = candidates.apply(
        get_source_names,
        axis=1,
    )

    candidates["source_count"] = candidates[
        list(source_columns.values())
    ].sum(axis=1).astype(int)

    # Put identifier and summary fields first
    ordered_columns = (
        ["candidate_id", "source_count", "source_results"]
        + list(source_columns.values())
        + ["geom"]
    )

    return candidates[ordered_columns]

Use it like this:

In [25]:
possible_missed_labels = create_missed_label_candidates(
    ref_gdf=ref_data_filtered,
    result_gdfs=result_layers,
)

print(f"Possible missed features: {len(possible_missed_labels)}")

Possible missed features: 7421


In [26]:
possible_missed_labels["source_count"].value_counts()

source_count
1    5439
2     547
6     404
3     350
4     255
5     250
7     176
Name: count, dtype: int64

# Save results

Save as GPKG file:

In [ ]:
possible_missed_labels.to_file(
    r"r:\delovno\nejc\26-7 model assisted labeling\possible_missed.gpkg",
    driver="GPKG",
)

Save directly to PostGIS

In [27]:
possible_missed_labels.to_postgis(
    name="test_upload",
    con=engine,
    schema="als_ml_rezultati",
    if_exists="replace",
    index=False,
)